# 01 A0 Protocol Audit

Generated by `scripts/revision/create_revision_notebooks.py`.


## Setup

In [ ]:
from pathlib import Path
import json
import os
import subprocess

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print(f'Drive mount skipped or already mounted: {exc}')

DRIVE_ROOT = Path('/content/drive/MyDrive/ECG-Ramba')
REPO_URL = 'https://github.com/BrianNguyen29/ECG-RAMBA.git'
BRANCH = os.environ.get('ECG_RAMBA_BRANCH', 'main')
REPO_DIR = Path(os.environ.get('ECG_RAMBA_REPO_DIR', '/content/ECG-RAMBA'))
MIRROR_REVISION_ROOT = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'
LOCAL_RUNTIME_ROOT = Path('/content/ecg_ramba_runtime')

os.environ['ECG_RAMBA_DRIVE_ROOT'] = str(DRIVE_ROOT)
os.environ.setdefault('ECG_RAMBA_LOCAL_ROOT', str(LOCAL_RUNTIME_ROOT))
os.environ.setdefault('ECG_RAMBA_EXTRACT_DIR', str(LOCAL_RUNTIME_ROOT / 'chapman'))
os.environ.setdefault('ECG_RAMBA_USE_CLEAN_CACHE', '0')
os.environ.setdefault('ECG_RAMBA_SAVE_CLEAN_CACHE', '0')

def _run_setup(cmd, cwd=None, check=True):
    print(f'$ {cmd}', flush=True)
    result = subprocess.run(
        cmd,
        shell=True,
        cwd=str(cwd) if cwd else None,
        check=False,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.stdout:
        print(result.stdout.rstrip())
    if check and result.returncode:
        raise subprocess.CalledProcessError(result.returncode, cmd, output=result.stdout)
    return result

def run_logged(cmd, log_path, cwd=None, check=True):
    run_cwd = Path(cwd) if cwd else Path.cwd()
    log_path = Path(log_path)
    if not log_path.is_absolute():
        log_path = run_cwd / log_path
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print(f'$ {cmd}', flush=True)
    with log_path.open('w', encoding='utf-8') as log:
        proc = subprocess.Popen(
            cmd, shell=True, cwd=str(run_cwd), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace', bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        return_code = proc.wait()
    print('Command log:', log_path)
    if check and return_code:
        raise subprocess.CalledProcessError(return_code, cmd)
    return subprocess.CompletedProcess(cmd, return_code)

def _git_setup(cmd, check=True):
    return _run_setup(cmd, cwd=REPO_DIR, check=check)

def _git_current_commit():
    result = _git_setup('git rev-parse --short HEAD', check=False)
    return result.stdout.strip() if result.returncode == 0 and result.stdout else 'unknown'

def _pull_or_continue(branch):
    before = _git_current_commit()
    status_result = _git_setup('git status --porcelain', check=False)
    status = status_result.stdout.strip() if status_result.stdout else ''
    if status:
        print('Local repo has changes before pull:')
        print(status[:4000])
    result = _git_setup(f'git pull --ff-only --autostash origin {branch}', check=False)
    after = _git_current_commit()
    if result.returncode:
        print('')
        print('=' * 80)
        print('WARNING: git pull failed; continuing with the currently checked-out repo.')
        print('This avoids deleting Drive files while long-running artifacts may exist.')
        print(f'Current commit: {after} (before pull: {before})')
        print('To force a clean code checkout later, rename the Drive repo folder or use a fresh clone.')
        print('=' * 80)
        if os.environ.get('ECG_RAMBA_ALLOW_STALE_REPO', '0') != '1':
            raise RuntimeError('git pull failed; refusing to continue with stale code. Set ECG_RAMBA_ALLOW_STALE_REPO=1 only for debugging.')
    else:
        print(f'Git update OK: {before} -> {after}')

if (REPO_DIR / '.git').exists():
    os.chdir(REPO_DIR)
    fetch_result = _git_setup('git fetch origin', check=False)
    if fetch_result.returncode:
        print('WARNING: git fetch failed; continuing with the currently checked-out repo.')
    current_branch_result = _git_setup('git branch --show-current', check=False)
    current_branch = current_branch_result.stdout.strip() if current_branch_result.stdout else ''
    if current_branch != BRANCH:
        checkout_result = _git_setup(f'git checkout {BRANCH}', check=False)
        if checkout_result.returncode:
            print(f'WARNING: git checkout {BRANCH} failed; continuing on branch {current_branch or "<detached>"}')
        else:
            current_branch = BRANCH
    if fetch_result.returncode == 0:
        _pull_or_continue(BRANCH)
elif (REPO_DIR / 'configs' / 'config.py').exists():
    os.chdir(REPO_DIR)
    print('Repo directory exists but is not a git checkout; skipping git pull.')
elif Path.cwd().joinpath('configs', 'config.py').exists():
    REPO_DIR = Path.cwd()
    os.chdir(REPO_DIR)
    if (REPO_DIR / '.git').exists():
        fetch_result = _run_setup('git fetch origin', check=False)
        if fetch_result.returncode == 0:
            _pull_or_continue(BRANCH)
        else:
            print('WARNING: git fetch failed; continuing with the currently checked-out repo.')
else:
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    _run_setup(f'git clone -b {BRANCH} {REPO_URL} {REPO_DIR}')
    os.chdir(REPO_DIR)

os.chdir(REPO_DIR)
_run_setup('git status --short --branch', check=False)

print('cwd       :', Path.cwd())
print('drive_root:', DRIVE_ROOT)
print('repo_dir  :', REPO_DIR)
print('branch    :', BRANCH)

cache_status = {
    'clean_raw_cache': DRIVE_ROOT / 'ecg_data_27c_subject.npz',
    'raw_minirocket_cache': DRIVE_ROOT / 'rocket_raw_N44186_C12_L5000_K10000_S42.npz',
    'hrv36_cache': DRIVE_ROOT / 'hrv36_N44186_C12_L5000.npz',
    'fold_pca_cache_dir': DRIVE_ROOT / 'revision_feature_cache',
}
print('cache policy:')
print('  ECG_RAMBA_USE_CLEAN_CACHE =', os.environ.get('ECG_RAMBA_USE_CLEAN_CACHE'))
print('  ECG_RAMBA_SAVE_CLEAN_CACHE=', os.environ.get('ECG_RAMBA_SAVE_CLEAN_CACHE'))
print('  ECG_RAMBA_EXTRACT_DIR     =', os.environ.get('ECG_RAMBA_EXTRACT_DIR'))
print('cache status:')
for name, path in cache_status.items():
    if path.is_dir():
        count = len(list(path.glob('*.npz')))
        print(f'  {name}: exists=True npz_count={count} path={path}')
    else:
        print(f'  {name}: exists={path.exists()} size={path.stat().st_size if path.exists() else None} path={path}')


## Revision Plan Files

In [ ]:
import pandas as pd

for path in [
    'docs/revision_plan/task_board.csv',
    'docs/revision_plan/claim_evidence_map.csv',
    'docs/revision_plan/experiment_registry.csv',
    'docs/revision_plan/a0_resolution_checklist.csv',
]:
    print('\n' + path)
    display(pd.read_csv(path).head(20))


## Run Protocol Audit

In [ ]:
run_logged(
    'python -u scripts/revision/00_audit_protocol.py',
    'reports/revision/logs/a0_protocol_audit.log',
)


## Inspect Audit Warnings

In [ ]:
audit = json.loads(Path('reports/revision/audit_protocol.json').read_text(encoding='utf-8'))
print('Warnings:')
for warning in audit.get('warnings', []):
    print('-', warning)

print('\nDataset paths:')
for key, info in audit['artifacts'].get('dataset_paths', {}).items():
    print(f"{key}: exists={info['exists']} path={info['path']}")


## Validate A0 Resolution Decisions

In [ ]:
run_logged(
    'python -u scripts/revision/00_a0_resolution_gate.py',
    'reports/revision/logs/a0_resolution_gate.log',
)


## Inspect A0 Completion Gate

In [ ]:
a0_status = json.loads(Path('reports/revision/a0_resolution_status.json').read_text(encoding='utf-8'))
print('A0 status        :', a0_status['status'])
print('Audit complete   :', a0_status['audit_complete'])
print('Protocol ready   :', a0_status['protocol_ready'])
print('Deferred blockers:', a0_status['deferred_blocker_ids'])
print('Meaning:', a0_status['meaning'])
for blocker in a0_status['blockers']:
    print(
        blocker['blocker_id'],
        blocker['resolution_status'],
        'valid=' + str(blocker['valid']),
        blocker['restriction'],
    )


## Inventory Current Artifacts

In [ ]:
run_logged(
    'python -u scripts/revision/05_artifact_inventory.py',
    'reports/revision/logs/a0_artifact_inventory.log',
)
run_logged(
    f'python -u scripts/revision/artifact_mirror.py publish --mirror-root "{MIRROR_REVISION_ROOT}"',
    'reports/revision/logs/a0_mirror_publish.log',
)
